# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Answer
The fastest growing continent is Africa, with a population growth of about 290.4%. This implies that the continent’s population grew to nearly four times what it was in 1970.

**Choice of chart and explanation:** A horizontal bar chart was chosen since it makes it easy to compare population growth percentages of continents.

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np
df = pd.read_csv("world_population.csv")

print(df.head())
print(df.columns.tolist())

   Rank CCA3 Country/Territory           Capital Continent  2022 Population  \
0    36  AFG       Afghanistan             Kabul      Asia         41128771   
1   138  ALB           Albania            Tirana    Europe          2842321   
2    34  DZA           Algeria           Algiers    Africa         44903225   
3   213  ASM    American Samoa         Pago Pago   Oceania            44273   
4   203  AND           Andorra  Andorra la Vella    Europe            79824   

   2020 Population  2015 Population  2010 Population  2000 Population  \
0         38972230         33753499         28189672         19542982   
1          2866849          2882481          2913399          3182021   
2         43451666         39543154         35856344         30774621   
3            46189            51368            54849            58230   
4            77700            71746            71519            66097   

   1990 Population  1980 Population  1970 Population  Area (km²)  \
0         10694796

In [3]:
# Q1: Which continent is growing fastest?

continent_growth = (
    df.groupby("Continent")
    .agg({
        "1970 Population": "sum",
        "2022 Population": "sum"
    })
    .reset_index()
)

continent_growth["Growth (%)"] = (
    (continent_growth["2022 Population"] -
     continent_growth["1970 Population"]) /
    continent_growth["1970 Population"] * 100
)

continent_growth = continent_growth.sort_values(
    "Growth (%)", ascending=True
)

fig = px.bar(
    continent_growth,
    x="Growth (%)",
    y="Continent",
    orientation="h",
    text_auto=".1f",
    title="Relative Population Growth by Continent (1970–2022)",
    template="plotly_white",
    color="Growth (%)",
    color_continuous_scale="Teal"
)

fig.update_layout(
    showlegend=False,
    xaxis_title="Population growth (%)",
    yaxis_title=None
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Answer:** India witnessed the largest increase in the period from 1970 to 2022, with about 860 million people, followed by China with roughly 600 million people. Out of the 15 nations, four belong to Africa, namely, Nigeria, Ethiopia, DRC, and Egypt.

**Choice of graph and reason:** The horizontal bar graph was chosen since it will make the comparison of population changes simple among countries. African countries have been marked in blue, while others are represented in grey.

In [4]:
# Q2: Which countries added the most people?

df["Population Added"] = (
    df["2022 Population"] - df["1970 Population"]
)

top15 = df.nlargest(15, "Population Added").copy()

# Count African countries in the top 15
africa_count = (top15["Continent"] == "Africa").sum()

print("Top 15 countries by population added:")
print(top15[["Country/Territory", "Population Added", "Continent"]])

print("\nNumber of African countries in top 15:", africa_count)

# Highlight African countries
top15["Region"] = top15["Continent"].apply(
    lambda x: "Africa" if x == "Africa" else "Other continents"
)

top15 = top15.sort_values("Population Added", ascending=True)

fig = px.bar(
    top15,
    x="Population Added",
    y="Country/Territory",
    orientation="h",
    color="Region",
    color_discrete_map={
        "Africa": "#0072B2",
        "Other continents": "#B8B8B8"
    },
    text_auto=".2s",
    title="15 Countries with the Largest Population Increase (1970–2022)",
    template="plotly_white"
)

fig.update_layout(
    xaxis_title="People added",
    yaxis_title=None,
    legend_title=None
)

fig.show()

Top 15 countries by population added:
    Country/Territory  Population Added      Continent
92              India         859671872           Asia
41              China         603352887           Asia
156          Pakistan         176533990           Asia
149           Nigeria         162971948         Africa
93          Indonesia         160272945           Asia
221     United States         137961517  North America
27             Brazil         118943623  South America
16         Bangladesh         103644512           Asia
63           Ethiopia          95071678         Africa
55           DR Congo          78858479         Africa
163       Philippines          78123423           Asia
131            Mexico          77214819  North America
57              Egypt          76208117         Africa
94               Iran          60100865           Asia
228           Vietnam          56258007           Asia

Number of African countries in top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Answer:** The relationship between the land area of a nation and the population of that nation is positively moderate, as the correlation coefficient is 0.453. But the large size of the land does not necessarily mean the large population size. Macau, Monaco and Singapore are good examples of nations with small land area but dense population. 

**Choice of the chart with reasons:** Scatter plot is chosen as the chart because it depicts well the relationship between two numerical variables – the size of the land area and population. The continents are distinguished by colour and the log scales help us compare countries of different sizes and population.

In [5]:

# Q3 - Do bigger countries have more people?

import plotly.express as px
import pandas as pd

# Calculate population density
df["Population Density"] = (
    df["2022 Population"] / df["Area (km²)"]
)

# Identify countries with unusually high population density
top_density = df.nlargest(5, "Population Density")
bottom_density = df.nsmallest(5, "Population Density")

# Create labels for notable countries
df["Label"] = ""
df.loc[top_density.index, "Label"] = top_density["Country/Territory"]
df.loc[bottom_density.index, "Label"] = bottom_density["Country/Territory"]

# Create scatter plot
fig = px.scatter(
    df,
    x="Area (km²)",
    y="2022 Population",
    color="Continent",
    hover_name="Country/Territory",
    hover_data=["Population Density"],
    text="Label",
    log_x=True,
    log_y=True,
    template="plotly_white",
    title="Land Area Alone Does Not Explain Population Size"
)

fig.update_traces(
    marker=dict(size=9, opacity=0.75),
    textposition="top center"
)

fig.update_layout(
    showlegend=True,
    xaxis_title="Land Area (km²)",
    yaxis_title="Population (log scale)",
    legend_title="Continent"
)

fig.show()

# Calculate correlation
correlation = df["Area (km²)"].corr(df["2022 Population"])

print("Correlation between land area and population:",
      round(correlation, 3))

print("\nFive most densely populated countries:")
print(top_density[["Country/Territory", "Population Density"]])

print("\nFive least densely populated countries:")
print(bottom_density[["Country/Territory", "Population Density"]])


Correlation between land area and population: 0.453

Five most densely populated countries:
    Country/Territory  Population Density
119             Macau        23172.266667
134            Monaco        18234.500000
187         Singapore         8416.463380
89          Hong Kong         6783.392210
76          Gibraltar         5441.500000

Five least densely populated countries:
    Country/Territory  Population Density
78          Greenland            0.026068
64   Falkland Islands            0.310523
230    Western Sahara            2.165361
135          Mongolia            2.172715
141           Namibia            3.109212


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Answer:** There is great variation in population density between and within continents. The continent with the highest median density is North America, with a population density of 221.42 per km2. This is followed by Asia and Europe at 122.97 and 106.87 people per km2 respectively. South America has the lowest median density, which stands at 17.79 people per km2. 

**Graph and its rationale:** Box plot was chosen since it highlights the distribution, median, dispersion and outliers of population density on different continents. Logarithmic scale was utilized so that comparisons could be made between densely and thinly populated countries.

In [6]:

# Q4 - How crowded are the continents?

import plotly.express as px
import pandas as pd

# Create box plot to compare population density
fig = px.box(
    df,
    x="Continent",
    y="Density (per km²)",
    color="Continent",
    points="outliers",
    log_y=True,
    template="plotly_white",
    title="Population Density Varies Widely Across Continents"
)

# Improve chart design
fig.update_layout(
    showlegend=False,
    xaxis_title=None,
    yaxis_title="Population Density (people per km²)",
    boxmode="group"
)

fig.show()

# Calculate statistics by continent
density_stats = df.groupby("Continent")["Density (per km²)"].agg(
    ["median", "mean", "min", "max"]
).round(2)

print("Population Density Statistics by Continent:")
print(density_stats)


Population Density Statistics by Continent:
               median     mean   min       max
Continent                                     
Africa          59.71   125.05  2.17    871.93
Asia           122.97  1025.02  2.17  23172.27
Europe         106.87   663.32  3.62  18234.50
North America  221.42   272.76  0.03   1299.26
Oceania         78.25   132.54  3.40    603.24
South America   17.79    20.97  0.31     65.02


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Answer:** In the period between 2000 and 2022, Qatar had the highest population growth rate at about 317.24%, followed by the United Arab Emirates at 188.25% and Equatorial Guinea at 144.52%. There were some countries that had population decline, especially small island countries and European countries.

**Choice of chart and explanation:** I have used a horizontal bar graph since it is easy to compare population growth and decline among countries. Bars above the x-axis show population growth, while those below the x-axis show population decline. Colors are used to show continents.

In [7]:

# Q5 - Where did populations grow and shrink?

import pandas as pd
import plotly.express as px

# Calculate percentage population change from 2000 to 2022
df["Population Change (%)"] = (
    (df["2022 Population"] - df["2000 Population"])
    / df["2000 Population"]
) * 100

# Get 10 countries with the highest population growth
top_growth = df.nlargest(10, "Population Change (%)")

# Get 10 countries with the largest population decline
top_decline = df.nsmallest(10, "Population Change (%)")

# Combine both groups
selected = pd.concat([top_growth, top_decline])

# Sort countries for visualization
selected = selected.sort_values("Population Change (%)")

# Create horizontal bar chart
fig = px.bar(
    selected,
    x="Population Change (%)",
    y="Country/Territory",
    color="Continent",
    orientation="h",
    hover_data=["2000 Population", "2022 Population"],
    template="plotly_white",
    title="Population Growth and Decline Across Countries (2000–2022)"
)

fig.update_layout(
    xaxis_title="Population Change (%)",
    yaxis_title=None,
    showlegend=True,
    legend_title="Continent"
)

fig.add_vline(x=0, line_dash="dash", line_color="grey")

fig.show()

# Display results
print("TOP 10 COUNTRIES WITH POPULATION GROWTH")
print(top_growth[
    ["Country/Territory", "Continent", "Population Change (%)"]
].round(2).to_string(index=False))

print("\nTOP 10 COUNTRIES WITH POPULATION DECLINE")
print(top_decline[
    ["Country/Territory", "Continent", "Population Change (%)"]
].round(2).to_string(index=False))


TOP 10 COUNTRIES WITH POPULATION GROWTH
       Country/Territory     Continent  Population Change (%)
                   Qatar          Asia                 317.24
    United Arab Emirates          Asia                 188.25
       Equatorial Guinea        Africa                 144.52
Turks and Caicos Islands North America                 143.83
                   Niger        Africa                 125.49
                  Jordan          Asia                 123.21
                  Kuwait          Asia                 120.62
                  Angola        Africa                 117.08
                    Chad        Africa                 114.59
          Western Sahara        Africa                 113.03

TOP 10 COUNTRIES WITH POPULATION DECLINE
       Country/Territory Continent  Population Change (%)
Northern Mariana Islands   Oceania                 -38.32
          American Samoa   Oceania                 -23.97
               Lithuania    Europe                 -23.60
    

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Answer:** Population distribution in the world as of 2022 is not evenly distributed across the continents. Asia has the highest percentage of the world's population, followed by Africa. Larger countries in terms of their populations are represented by larger rectangles. This makes it easy to compare continental and country population distributions.

**Choice of chart and rationale:** Treemap was chosen for this chart since it represents hierarchical data in one chart. Continents are the top level of hierarchy and countries are the lower levels. Size of rectangles shows their population in 2022.

In [8]:

# Q6 - Where do the world's 8 billion people live?

import plotly.express as px
import pandas as pd

# Prepare the data
q6 = df.dropna(
    subset=["Continent", "Country/Territory", "2022 Population"]
).copy()

q6["2022 Population"] = pd.to_numeric(
    q6["2022 Population"], errors="coerce"
)

q6 = q6.dropna(subset=["2022 Population"])

# Exclude any world-level aggregate to avoid double counting
q6 = q6[q6["Country/Territory"] != "World"]

# Find the largest continent
continent_pop = q6.groupby("Continent")["2022 Population"].sum()
largest = continent_pop.idxmax()

# Create treemap
fig = px.treemap(
    q6,
    path=[
        px.Constant("World"),
        "Continent",
        "Country/Territory"
    ],
    values="2022 Population",
    color="Continent",
    color_discrete_sequence=[
        "#0072B2",
        "#E69F00",
        "#009E73",
        "#CC79A7",
        "#56B4E9",
        "#D55E00"
    ],
    title=f"{largest} Holds the Largest Share of World Population"
)

fig.update_traces(
    textinfo="label+percent parent",
    textfont_size=12,
    marker_line_width=1,
    marker_line_color="white"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    margin=dict(t=80, l=20, r=20, b=20),
    title_font_size=20
)

fig.show()

# Display population by continent
print("2022 Population by Continent:")
print(continent_pop.sort_values(ascending=False))


2022 Population by Continent:
Continent
Asia             4721383274
Africa           1426730932
Europe            743147538
North America     600296136
South America     436816608
Oceania            45038554
Name: 2022 Population, dtype: int64


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Solution:** Africa had the highest growth in the proportion of the world population, increasing from 9.89% to 17.89% from 1970 to 2022, an increase of 8 percentage points. There is a slight increase in Asia from 58.06% to 59.21%. Europe has the highest decrease, from 17.76% to 9.32%. There is also a slight decrease in “the rest of the world” from 14.29% to 13.57%.

**Type of graph and reasons for selection:** Line graph is selected since it helps in comparing changes in population proportions from 1970 to 2022. Colour difference distinguishes different regions.

In [9]:

# Q7 - Whose share of the world is rising?

import pandas as pd
import plotly.express as px
import numpy as np

# Calculate population totals for 1970 and 2022
q7 = df.copy()

# Group continents into the required categories
q7["Group"] = q7["Continent"].apply(
    lambda x: x if x in ["Africa", "Asia", "Europe"]
    else "Rest of the world"
)

# Calculate total population by group
pop_1970 = q7.groupby("Group")["1970 Population"].sum()
pop_2022 = q7.groupby("Group")["2022 Population"].sum()

# Combine results
shares = pd.DataFrame({
    "1970": pop_1970,
    "2022": pop_2022
})

# Calculate percentage share of world population
shares = shares.div(shares.sum(axis=0), axis=1) * 100

# Prepare chart data
chart_data = shares.reset_index().melt(
    id_vars="Group",
    var_name="Year",
    value_name="Share"
)

# Calculate change in population share
change = shares["2022"] - shares["1970"]

largest_gain = change.idxmax()
largest_drop = change.idxmin()

# Create line chart
fig = px.line(
    chart_data,
    x="Year",
    y="Share",
    color="Group",
    markers=True,
    text="Share",
    color_discrete_map={
        "Africa": "#009E73",
        "Asia": "#0072B2",
        "Europe": "#D55E00",
        "Rest of the world": "#CC79A7"
    },
    title=f"{largest_gain} Gains the Most World Population Share"
)

fig.update_traces(
    texttemplate="%{y:.1f}%",
    textposition="top center",
    line=dict(width=3),
    marker=dict(size=11)
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Share of world population (%)",
    xaxis_title=None,
    legend_title="Region",
    hovermode="x unified"
)

fig.show()

# Display results
print("Population share (%) by group:")
print(shares.round(2))

print("\nChange in population share (percentage points):")
print(change.round(2))


Population share (%) by group:
                    1970   2022
Group                          
Africa              9.89  17.89
Asia               58.06  59.21
Europe             17.76   9.32
Rest of the world  14.29  13.57

Change in population share (percentage points):
Group
Africa               8.00
Asia                 1.15
Europe              -8.44
Rest of the world   -0.72
dtype: float64
